In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf transformers accelerate sentence-transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving household_notes.pdf to household_notes.pdf


In [ ]:
import os

os.makedirs("./docs", exist_ok=True)

In [ ]:
import shutil

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        shutil.move(
            filename,
            os.path.join("./docs", filename)
        )

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_directory = "./docs"

documents = []

file_paths = [
    os.path.join(pdf_directory, file)
    for file in os.listdir(pdf_directory)
    if file.lower().endswith(".pdf")
]

for file_path in file_paths:

    loader = PyPDFLoader(file_path)

    documents.extend(loader.load())

print("Number of pages loaded:", len(documents))

Number of pages loaded: 4


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

text_splitted_document = text_splitter.split_documents(documents)

print("Number of chunks:", len(text_splitted_document))

Number of chunks: 18


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    text_splitted_document,
    embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [ ]:
evaluation_dataset = [

    # --------------------------------------------------
    # DIRECT FACTUAL QUESTIONS
    # --------------------------------------------------

    {
        "question": "What is the remaining mortgage balance?",
        "expected": "342,600",
        "category": "Direct"
    },

    {
        "question": "What is the monthly mortgage payment?",
        "expected": "2,180",
        "category": "Direct"
    },

    {
        "question": "What is the interest rate on the mortgage?",
        "expected": "5.35",
        "category": "Direct"
    },

    {
        "question": "How much remains on the car loan?",
        "expected": "11,400",
        "category": "Direct"
    },

    {
        "question": "What is the monthly car loan payment?",
        "expected": "365",
        "category": "Direct"
    },

    {
        "question": "How much is currently in the emergency fund?",
        "expected": "15,000",
        "category": "Direct"
    },

    {
        "question": "What is the target for the emergency fund?",
        "expected": "24,000",
        "category": "Direct"
    },

    {
        "question": "How much is currently in the brokerage account?",
        "expected": "58,300",
        "category": "Direct"
    },


    # --------------------------------------------------
    # PARAPHRASED QUESTIONS
    # --------------------------------------------------

    {
        "question": "How much does the household pay every month toward the home loan?",
        "expected": "2,180",
        "category": "Paraphrased"
    },

    {
        "question": "What amount has been set aside as the household's financial safety cushion?",
        "expected": "15,000",
        "category": "Paraphrased"
    },

    {
        "question": "What sum has been accumulated for repairing the house?",
        "expected": "6,200",
        "category": "Paraphrased"
    },

    {
        "question": "How much money has been put away for the eventual replacement of Priya's car?",
        "expected": "4,800",
        "category": "Paraphrased"
    },

    {
        "question": "What is the current value of the household's taxable investment account?",
        "expected": "58,300",
        "category": "Paraphrased"
    },


    # --------------------------------------------------
    # CONCEPTUAL QUESTIONS
    # --------------------------------------------------

    {
        "question": "Why does the household keep several savings accounts instead of putting all the money into one account?",
        "expected": "separate goals",
        "category": "Conceptual"
    },

    {
        "question": "Why is the emergency fund kept separate from the home repair fund?",
        "expected": "emergencies",
        "category": "Conceptual"
    },

    {
        "question": "Why do Daniel and Priya consider the car loan more urgent than the mortgage?",
        "expected": "higher rate",
        "category": "Conceptual"
    },

    {
        "question": "Why don't they consider their brokerage account to be emergency savings?",
        "expected": "market",
        "category": "Conceptual"
    },

    {
        "question": "Why is the vacation fund considered more flexible than the vehicle fund?",
        "expected": "vacation can be postponed",
        "category": "Conceptual"
    },


    # --------------------------------------------------
    # COMPARISON QUESTIONS
    # --------------------------------------------------

    {
        "question": "Which has the higher interest rate, the mortgage or the car loan?",
        "expected": "car loan",
        "category": "Comparison"
    },

    {
        "question": "How does the purpose of the emergency fund differ from the home repair fund?",
        "expected": "emergency fund",
        "category": "Comparison"
    },

    {
        "question": "How are the brokerage account and cryptocurrency holdings treated differently?",
        "expected": "cryptocurrency",
        "category": "Comparison"
    },

    {
        "question": "Which is considered more urgent, replacing the car or taking the vacation?",
        "expected": "vehicle",
        "category": "Comparison"
    },


    # --------------------------------------------------
    # MULTI-HOP QUESTIONS
    # --------------------------------------------------

    {
        "question": "Why do they prioritize paying off the car loan before making extra mortgage payments?",
        "expected": "car loan",
        "category": "Multi-hop"
    },

    {
        "question": "What financial priority comes after finishing the car loan?",
        "expected": "emergency fund",
        "category": "Multi-hop"
    },

    {
        "question": "Why does the household continue retirement contributions even while dealing with shorter-term goals?",
        "expected": "long-term",
        "category": "Multi-hop"
    },


    # --------------------------------------------------
    # MISSING INFORMATION / HALLUCINATION TESTS
    # --------------------------------------------------

    {
        "question": "What is the name of the institution that currently services the mortgage?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What is the current balance of the college fund?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What specific car model does Priya plan to buy?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What is their planned vacation destination?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },

    {
        "question": "What medications are Daniel and Priya currently taking?",
        "expected": "I couldn't find this information",
        "category": "Missing Information"
    },


    # --------------------------------------------------
    # NUMERICAL / CALCULATION QUESTIONS
    # --------------------------------------------------

    {
        "question": "How much more money is needed to reach the emergency fund target?",
        "expected": "9,000",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed to reach the home repair fund target?",
        "expected": "3,800",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed to reach the vacation fund target?",
        "expected": "1,850",
        "category": "Calculation"
    },

    {
        "question": "How much more is needed for the vehicle savings account to reach its target?",
        "expected": "7,200",
        "category": "Calculation"
    }
]

In [ ]:
for i, test in enumerate(evaluation_dataset):

    query = test["question"]

    result = retriever.invoke(query)

    print("\n" + "=" * 70)
    print(f"QUESTION {i + 1}")
    print("=" * 70)

    print("Question:", query)
    print("Category:", test["category"])

    print("\nRetrieved Chunks:")

    for j, doc in enumerate(result):
        print(f"\n--- Chunk {j + 1} ---")
        print(doc.page_content)


QUESTION 1
Question: What is the remaining mortgage balance?
Category: Direct

Retrieved Chunks:

--- Chunk 1 ---
to the top of their list of things to deal with. The escrow portion of their payment covers both property
taxes and homeowners insurance, though the notes here do not break down what share of the 2,180
dollar payment goes toward each piece versus principal and interest.
The Car Loan
Separately, the household is still paying off an auto loan with a remaining balance of 11,400 dollars.
The interest rate on this loan is 6.9 percent, notably higher than the mortgage rate, and the monthly
payment is 365 dollars. Because the balance is comparatively small and the rate is high, Daniel and
Priya have agreed that this is the debt they want gone first, and at the current payment pace they
expect it to be fully retired within roughly fourteen months. Unlike the mortgage, there is no long
amortization horizon to think about here, which is part of why it feels more urgent to them even


In [ ]:
from transformers import pipeline
import torch

llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    max_new_tokens=150,
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
def ask_rag(question, context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise question-answering assistant. "
                "Answer ONLY using the information in the provided context. "
                "If the answer is not in the context, respond exactly with: "
                "\"I couldn't find this information.\" "
                "Do not guess, infer, or add any detail not explicitly stated in the context. "
                "Be concise and directly answer what is asked."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    output = llm_pipeline(
        messages,
        max_new_tokens=150,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

    return output[0]["generated_text"][-1]["content"]

In [ ]:
import time

def run_rag_evaluation(evaluation_dataset, retriever, ask_rag_fn, verbose=True):
    """
    evaluation_dataset : list of dicts, each with "question", "expected", "category"
    retriever          : function(question) -> context string (your existing retrieval step)
    ask_rag_fn         : the ask_rag(question, context) function defined earlier
    """
    results = []

    for i, item in enumerate(evaluation_dataset, start=1):
        question = item["question"]
        expected = item.get("expected", "N/A")
        category = item.get("category", "Uncategorized")


        context = retriever(question)


        start = time.time()
        answer = ask_rag_fn(question, context)
        elapsed = time.time() - start

        result = {
            "index": i,
            "category": category,
            "question": question,
            "expected": expected,
            "context": context,
            "model_answer": answer,
            "time_sec": round(elapsed, 2),
        }
        results.append(result)

        if verbose:
            print("=" * 70)
            print(f"QUESTION {i} [{category}]")
            print("=" * 70)
            print(f"Question: {question}")
            print(f"Expected: {expected}")
            print(f"Model Answer: {answer}")
            print(f"Time: {elapsed:.2f}s")
            print()

    return results


def summarize_results(results, save_csv_path=None):
    """
    Prints a per-category and overall breakdown.
    Does NOT auto-grade (substring matching is unreliable, as we found) —
    it just organizes results so you can eyeball each one against the source doc.
    """
    from collections import defaultdict

    by_category = defaultdict(list)
    for r in results:
        by_category[r["category"]].append(r)

    print("\n" + "=" * 70)
    print("SUMMARY BY CATEGORY")
    print("=" * 70)
    for category, items in by_category.items():
        print(f"\n{category} ({len(items)} questions):")
        for r in items:
            print(f"  Q{r['index']}: {r['question'][:60]}...")
            print(f"    -> {r['model_answer'][:100]}")

    if save_csv_path:
        import csv
        with open(save_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["index", "category", "question", "expected", "model_answer", "time_sec"])
            writer.writeheader()
            for r in results:
                writer.writerow({k: r[k] for k in writer.fieldnames})
        print(f"\nSaved results to {save_csv_path}")

    return by_category




In [ ]:
def get_context(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return context


In [ ]:
results = run_rag_evaluation(
    evaluation_dataset=evaluation_dataset,
    retriever=get_context,
    ask_rag_fn=ask_rag,
    verbose=True
)

summarize_results(results, save_csv_path="rag_results.csv")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 1 [Direct]
Question: What is the remaining mortgage balance?
Expected: 342,600
Model Answer: 342,600 dollars
Time: 2.48s



[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 2 [Direct]
Question: What is the monthly mortgage payment?
Expected: 2,180
Model Answer: 2,180 dollars
Time: 1.61s



[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 32 [Calculation]
Question: How much more is needed to reach the home repair fund target?
Expected: 3,800
Model Answer: 3800 dollars
Time: 1.53s



[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 33 [Calculation]
Question: How much more is needed to reach the vacation fund target?
Expected: 1,850
Model Answer: 1,850 dollars
Time: 1.67s

QUESTION 34 [Calculation]
Question: How much more is needed for the vehicle savings account to reach its target?
Expected: 7,200
Model Answer: 7200 dollars.
Time: 1.68s


SUMMARY BY CATEGORY

Direct (8 questions):
  Q1: What is the remaining mortgage balance?...
    -> 342,600 dollars
  Q2: What is the monthly mortgage payment?...
    -> 2,180 dollars
  Q3: What is the interest rate on the mortgage?...
    -> 5.35 percent
  Q4: How much remains on the car loan?...
    -> 11,400 dollars
  Q5: What is the monthly car loan payment?...
    -> The monthly car loan payment is 365 dollars.
  Q6: How much is currently in the emergency fund?...
    -> 15,000 dollars
  Q7: What is the target for the emergency fund?...
    -> The target for the emergency fund is 24,000 dollars.
  Q8: How much is currently in the brokerage account?...
    -> 58,300

defaultdict(list,
            {'Direct': [{'index': 1,
               'category': 'Direct',
               'question': 'What is the remaining mortgage balance?',
               'expected': '342,600',
               'context': "to the top of their list of things to deal with. The escrow portion of their payment covers both property\ntaxes and homeowners insurance, though the notes here do not break down what share of the 2,180\ndollar payment goes toward each piece versus principal and interest.\nThe Car Loan\nSeparately, the household is still paying off an auto loan with a remaining balance of 11,400 dollars.\nThe interest rate on this loan is 6.9 percent, notably higher than the mortgage rate, and the monthly\npayment is 365 dollars. Because the balance is comparatively small and the rate is high, Daniel and\nPriya have agreed that this is the debt they want gone first, and at the current payment pace they\nexpect it to be fully retired within roughly fourteen months. Unlike the mort

# **Evaluation Harness**

Performing three different kinds of chunking to get the best accuracy results

In [ ]:
!pip install -q langchain langchain-community langchain-experimental langchain-text-splitters sentence-transformers faiss-cpu pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter


loader = PyPDFLoader("/content/docs/household_notes.pdf")
raw_documents = loader.load()


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_822/2210563456.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Fixed Chunking

In [ ]:
def build_fixed_chunking_retriever(chunk_size=500, k=4):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Fixed Chunking] Created {len(chunks)} chunks (size={chunk_size}, overlap=0)")
    return retriever


fixed_retriever = build_fixed_chunking_retriever(chunk_size=500, k=4)

[Fixed Chunking] Created 30 chunks (size=500, overlap=0)


# Overlap Chunking

In [ ]:
def build_overlap_chunking_retriever(chunk_size=500, chunk_overlap=100, k=4):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    chunks = splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Overlap Chunking] Created {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")
    return retriever


overlap_retriever = build_overlap_chunking_retriever(chunk_size=500, chunk_overlap=100, k=4)

[Overlap Chunking] Created 35 chunks (size=500, overlap=100)


# Semantic Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

def build_semantic_chunking_retriever(breakpoint_threshold_amount=80, k=4):
    semantic_splitter = SemanticChunker(
        embedding_model,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=breakpoint_threshold_amount,
    )
    chunks = semantic_splitter.split_documents(raw_documents)

    vectorstore = FAISS.from_documents(chunks, embedding_model)
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    print(f"[Semantic Chunking] Created {len(chunks)} chunks")
    return retriever


semantic_retriever = build_semantic_chunking_retriever(breakpoint_threshold_amount=80, k=4)

[Semantic Chunking] Created 17 chunks


In [ ]:
def get_context_fn(retriever_obj):
    """Wraps any LangChain retriever into the get_context(question) function
    that run_rag_evaluation expects."""
    def get_context(question):
        docs = retriever_obj.invoke(question)
        return "\n\n".join([doc.page_content for doc in docs])
    return get_context


strategies = {
    "fixed": fixed_retriever,
    "overlap": overlap_retriever,
    "semantic": semantic_retriever,
}

all_results = {}

for name, retriever_obj in strategies.items():
    print(f"\n{'#'*70}\nRunning evaluation with: {name.upper()} chunking\n{'#'*70}")
    results = run_rag_evaluation(
        evaluation_dataset=evaluation_dataset,
        retriever=get_context_fn(retriever_obj),
        ask_rag_fn=ask_rag,
        verbose=False,
    )
    summarize_results(results, save_csv_path=f"rag_results_{name}.csv")
    all_results[name] = results

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



######################################################################
Running evaluation with: FIXED chunking
######################################################################


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_len


SUMMARY BY CATEGORY

Direct (8 questions):
  Q1: What is the remaining mortgage balance?...
    -> 342,600 dollars
  Q2: What is the monthly mortgage payment?...
    -> 2,180 dollars
  Q3: What is the interest rate on the mortgage?...
    -> 5.35 percent
  Q4: How much remains on the car loan?...
    -> 11,400 dollars
  Q5: What is the monthly car loan payment?...
    -> 1140
  Q6: How much is currently in the emergency fund?...
    -> 15,000 dollars
  Q7: What is the target for the emergency fund?...
    -> The target for the emergency fund is 24,000 dollars.
  Q8: How much is currently in the brokerage account?...
    -> 58,300 dollars

Paraphrased (5 questions):
  Q9: How much does the household pay every month toward the home ...
    -> 365 dollars
  Q10: What amount has been set aside as the household's financial ...
    -> 15,000 dollars
  Q11: What sum has been accumulated for repairing the house?...
    -> 6,200 dollars
  Q12: How much money has been put away for the eventual 

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


SUMMARY BY CATEGORY

Direct (8 questions):
  Q1: What is the remaining mortgage balance?...
    -> 342,600 dollars
  Q2: What is the monthly mortgage payment?...
    -> 2,180 dollars
  Q3: What is the interest rate on the mortgage?...
    -> I couldn't find this information.
  Q4: How much remains on the car loan?...
    -> 11,400 dollars
  Q5: What is the monthly car loan payment?...
    -> 365 dollars
  Q6: How much is currently in the emergency fund?...
    -> 15,000 dollars
  Q7: What is the target for the emergency fund?...
    -> The target for the emergency fund is 24,000 dollars.
  Q8: How much is currently in the brokerage account?...
    -> 58,300 dollars

Paraphrased (5 questions):
  Q9: How much does the household pay every month toward the home ...
    -> 2,180 dollars
  Q10: What amount has been set aside as the household's financial ...
    -> 15,000 dollars
  Q11: What sum has been accumulated for repairing the house?...
    -> 6,200 dollars
  Q12: How much money has b

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


SUMMARY BY CATEGORY

Direct (8 questions):
  Q1: What is the remaining mortgage balance?...
    -> The remaining mortgage balance is 342,600 dollars.
  Q2: What is the monthly mortgage payment?...
    -> The monthly mortgage payment is 2,180 dollars.
  Q3: What is the interest rate on the mortgage?...
    -> The interest rate on the mortgage is 5.35 percent.
  Q4: How much remains on the car loan?...
    -> 11,400 dollars remains on the car loan.
  Q5: What is the monthly car loan payment?...
    -> The monthly car loan payment is 365 dollars.
  Q6: How much is currently in the emergency fund?...
    -> 15,000 dollars
  Q7: What is the target for the emergency fund?...
    -> The target for the emergency fund is 24,000 dollars.
  Q8: How much is currently in the brokerage account?...
    -> 58,300 dollars

Paraphrased (5 questions):
  Q9: How much does the household pay every month toward the home ...
    -> 2,180 dollars
  Q10: What amount has been set aside as the household's financ